# FAseg — Batch Inference on Ex-Vivo Data (Pretraining Only, No Fine-tuning)

Runs inference for **5 pretraining experiments × best_s3 checkpoint** on
ex-vivo beef leg data — **no fine-tuning applied**.

Click "Run All".

Each run saves to `outputs/inference_results/exvivo/without_fine_tuning/<exp>_best_s3/`:
- `pred_mask_3d.npy` / `.bin` — 3-D binary mask
- `boundary_matrix.npy` — TOF boundary per row/slice
- `inference_time.txt`
- `source_info.txt` — which pretraining epoch was used

In [1]:
import os, sys, json, time
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

_NB_DIR = os.path.dirname(os.path.abspath(''))
if os.path.basename(_NB_DIR) == 'scripts':
    _project_root = os.path.dirname(_NB_DIR)
else:
    _project_root = _NB_DIR
_src_root = os.path.join(_project_root, 'src')
if _src_root not in sys.path:
    sys.path.insert(0, _src_root)

from faseg.models import UNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [2]:
# =========================================================================
# Config — all pretraining experiments (best_s3 checkpoint only)
# =========================================================================
DATA_DIR = os.path.join(_project_root, 'manual segmentation', 'manual_seg_beef')

# (pretrain_dir, label) — label is for display only; subfolder uses directory name
PRETRAIN_RUNS = [
    ('outputs/pretraining',                'Full'),
    ('outputs/pretraining_no_warmup',      'No Warmup'),
    ('outputs/pretraining_no_tof',         'No TOF'),
    ('outputs/pretraining_no_domainrand',  'No DomainRand'),
    ('outputs/pretraining_no_aug',         'No Aug'),
]

STRATEGY = 'best_s3'
N_S3_START = 61
SLICE_X = 32  # which cross-section to use per src

print(f'Will run inference for {len(PRETRAIN_RUNS)} pretraining models on ex-vivo data ({STRATEGY})')

Will run inference for 5 pretraining models on ex-vivo data (best_s3)


In [3]:
# =========================================================================
# Load & preprocess ex-vivo data — one slice per odd src
# =========================================================================
def load_exvivo_stack(data_dir, slice_x=32):
    """Load ex-vivo .bin files for all odd src, build (S, H, W) stack.

    Only odd src indices that have BOTH a mask AND data file are
    loaded.  Missing src (e.g. src317) are filled with zeros.
    Returns (512, 384, 384) float32 array.
    """
    stack = np.zeros((512, 384, 384), dtype=np.float32)
    loaded, missing = 0, 0
    for src in range(1, 512, 2):  # odd only
        # Skip src that don't have a mask at all (e.g. src317)
        mask_path = os.path.join(data_dir, f'src{src}_mask.npy')
        if not os.path.exists(mask_path):
            missing += 1
            continue

        fname = f'slice{slice_x}_src{src}.bin'
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            print(f'  WARNING: {fname} not found — filling with zeros')
            missing += 1
            continue

        sub = np.fromfile(fpath, dtype=np.int16).reshape((384, 384)).astype(np.float32)
        vmax = np.abs(sub).max()
        if vmax > 0:
            sub /= vmax
        sub = np.clip(sub, -1.0, 1.0)
        stack[src - 1] = sub  # src1 → index 0
        loaded += 1
    print(f'  Loaded {loaded} slices, {missing} missing/zero-filled')
    return stack

SLICE_X = 32  # which cross-section to use per src
print(f'Loading ex-vivo data (slice {SLICE_X} per src) ...')
stack = load_exvivo_stack(DATA_DIR, slice_x=SLICE_X)
S, H, W = stack.shape
print(f'Preprocessed: {stack.shape}  range=[{stack.min():.3f}, {stack.max():.3f}]')
print(f'Nonzero slices: {(np.abs(stack).sum(axis=(1,2)) > 0).sum()}')

# DataLoader (reused for all models)
tensor_stack = torch.from_numpy(stack).unsqueeze(1)
dataset = TensorDataset(tensor_stack)
loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4)

Loading ex-vivo data (slice 32 per src) ...
  Loaded 255 slices, 1 missing/zero-filled
Preprocessed: (512, 384, 384)  range=[-0.999, 1.000]
Nonzero slices: 255


In [4]:
# =========================================================================
# Find best_s3 checkpoint + run inference
# =========================================================================
def find_best_s3_checkpoint(pretrain_dir):
    """Return (epoch, val_loss, ckpt_path) for best Stage-3 validation loss."""
    loss_log = os.path.join(pretrain_dir, 'loss_log.txt')
    if not os.path.exists(loss_log):
        raise FileNotFoundError(f'loss_log.txt not found in {pretrain_dir}')
    best_ep, best_val = None, float('inf')
    with open(loss_log) as f:
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 3:
                ep, val = int(parts[0]), float(parts[2])
                if ep >= N_S3_START and val < best_val:
                    best_val = val
                    best_ep = ep
    if best_ep is None:
        raise RuntimeError(f'No Stage-3 epoch found in {loss_log}')
    ckpt_path = os.path.join(pretrain_dir, f'unet_epoch{best_ep:02d}.pth')
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f'Checkpoint not found: {ckpt_path}')
    return best_ep, best_val, ckpt_path


def run_inference_for_model(save_dir, model, loader, S, H):
    """Run inference and save results to *save_dir*."""
    os.makedirs(save_dir, exist_ok=True)
    all_masks = []
    t0 = time.time()
    with torch.no_grad():
        for (batch,) in loader:
            batch = batch.to(device)
            logits = model(batch)
            preds = logits.argmax(dim=1).cpu().numpy().astype(np.uint8)
            all_masks.append(preds)
    pred_masks = np.concatenate(all_masks, axis=0)
    elapsed = time.time() - t0

    boundary_matrix = np.full((H, S), np.nan, dtype=np.float32)
    for s in range(S):
        for row in range(H):
            ones = np.where(pred_masks[s, row, :] == 1)[0]
            if len(ones) > 0:
                boundary_matrix[row, s] = float(ones[0])

    mask_3d = pred_masks.transpose(1, 2, 0)
    np.save(os.path.join(save_dir, 'pred_mask_3d.npy'), mask_3d)
    np.asfortranarray(mask_3d).tofile(os.path.join(save_dir, 'pred_mask_3d.bin'))
    np.save(os.path.join(save_dir, 'boundary_matrix.npy'), boundary_matrix)
    with open(os.path.join(save_dir, 'inference_time.txt'), 'w') as f:
        f.write(f'{elapsed:.2f}s  ({S} slices, {S/elapsed:.0f} slices/s)\n')
    return elapsed

In [5]:
# =========================================================================
# Run all
# =========================================================================
results = []
t_start = time.time()
INFERENCE_ROOT = os.path.join(_project_root, 'outputs', 'inference_results', 'exvivo', 'without_fine_tuning')

for pretrain_rel, label in PRETRAIN_RUNS:
    pretrain_dir = os.path.join(_project_root, pretrain_rel)

    exp_name = os.path.basename(pretrain_rel)
    save_subdir = f'{exp_name}_{STRATEGY}'
    save_dir = os.path.join(INFERENCE_ROOT, save_subdir)

    print(f'\n{"="*60}')
    print(f'  {label}  ({STRATEGY})  |  ex-vivo')
    print(f'  Pretrain : {pretrain_dir}')
    print(f'  Save to  : {save_dir}')
    print(f'{"="*60}')

    try:
        best_ep, best_val, ckpt_path = find_best_s3_checkpoint(pretrain_dir)
    except Exception as e:
        print(f'  !! SKIP: {e}')
        results.append((label, 'SKIP', str(e)))
        continue

    print(f'  Best S3 epoch  : {best_ep}  (val_loss={best_val:.4f})')

    config_path = os.path.join(pretrain_dir, 'config.txt')
    if os.path.exists(config_path):
        with open(config_path) as f:
            cfg = json.load(f)
        base_ch = cfg.get('base_channel', 64)
        dropout = cfg.get('dropout_prob', 0.2)
        use_bn  = cfg.get('use_bn', True)
    else:
        base_ch, dropout, use_bn = 64, 0.2, True

    model = UNet(in_ch=1, base_ch=base_ch, num_classes=2,
                 dropout_prob=dropout, use_bn=use_bn).to(device)
    raw = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(raw, dict) and 'model_state_dict' in raw:
        state = raw['model_state_dict']
    else:
        state = raw
    model.load_state_dict(state)
    model.eval()

    elapsed = run_inference_for_model(save_dir, model, loader, S, H)
    print(f'  DONE — {S} slices in {elapsed:.1f}s  ({S/elapsed:.0f} slices/s)')

    with open(os.path.join(save_dir, 'source_info.txt'), 'w') as f:
        f.write(f'pretrain_dir={pretrain_dir}\n')
        f.write(f'pretrain_epoch={best_ep}\n')
        f.write(f'pretrain_val_loss={best_val:.6f}\n')
        f.write(f'strategy={STRATEGY}\n')
        f.write(f'fine_tuning=False\n')
        f.write(f'dataset=exvivo (manual_seg_beef, slice_{SLICE_X})\n')

    results.append((label, f'{elapsed:.1f}s', best_ep))

# =========================================================================
# Summary
# =========================================================================
print(f'\n{"="*60}')
print(f'  SUMMARY  (total: {(time.time()-t_start)/60:.0f} min)')
print(f'{"="*60}')
for item in results:
    if len(item) == 3:
        lbl, status, detail = item
        if status == 'SKIP':
            print(f'  {lbl:25s}  SKIP  ({detail})')
        else:
            print(f'  {lbl:25s}  {status:>8s}  (epoch {detail})')
    else:
        print(f'  {item[0]:25s}  {item[1]}')


  Full  (best_s3)  |  ex-vivo
  Pretrain : /data/projects/AgentWork/FAseg for github/outputs/pretraining
  Save to  : /data/projects/AgentWork/FAseg for github/outputs/inference_results/exvivo/without_fine_tuning/pretraining_best_s3
  Best S3 epoch  : 84  (val_loss=0.0038)


/home/yifei-sun/anaconda3/envs/faseg_ablation/lib/python3.10/site-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5090 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_89 sm_90 compute_90.
If you want to use the NVIDIA GeForce RTX 5090 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


  DONE — 512 slices in 2.9s  (178 slices/s)

  No Warmup  (best_s3)  |  ex-vivo
  Pretrain : /data/projects/AgentWork/FAseg for github/outputs/pretraining_no_warmup
  Save to  : /data/projects/AgentWork/FAseg for github/outputs/inference_results/exvivo/without_fine_tuning/pretraining_no_warmup_best_s3
  Best S3 epoch  : 87  (val_loss=0.0038)
  DONE — 512 slices in 2.9s  (179 slices/s)

  No TOF  (best_s3)  |  ex-vivo
  Pretrain : /data/projects/AgentWork/FAseg for github/outputs/pretraining_no_tof
  Save to  : /data/projects/AgentWork/FAseg for github/outputs/inference_results/exvivo/without_fine_tuning/pretraining_no_tof_best_s3
  Best S3 epoch  : 92  (val_loss=0.0037)
  DONE — 512 slices in 2.8s  (183 slices/s)

  No DomainRand  (best_s3)  |  ex-vivo
  Pretrain : /data/projects/AgentWork/FAseg for github/outputs/pretraining_no_domainrand
  Save to  : /data/projects/AgentWork/FAseg for github/outputs/inference_results/exvivo/without_fine_tuning/pretraining_no_domainrand_best_s3
  Best